In [23]:
import pandas as pd
import numpy as np
import os
import requests
from tqdm import tqdm
from datetime import datetime, timedelta
import json
import re
from rapidfuzz import fuzz, process


In [25]:
with open("../data/raw/seasons.json","r") as f:
    seasons = json.load(f)

seasons_df=pd.DataFrame(seasons["seasons"])

seasons_df.to_csv("seasons_df.csv")
seasons_df = seasons_df[~seasons_df.name.str.contains("UTR")]
seasons_df = seasons_df[~seasons_df.name.str.contains("ITF")]
seasons_df = seasons_df[~seasons_df.name.str.contains("Challenger")]
seasons_df = seasons_df[~seasons_df.name.str.contains("WTA 125K")]
seasons_df = seasons_df[~seasons_df.name.str.contains("WTA 250K")]
seasons_df = seasons_df[(seasons_df.name.str.contains("ATP"))|(seasons_df.name.str.contains("WTA"))]




In [26]:
df = pd.concat([pd.read_csv("../data/raw/sackmann_atp_matches_2024.csv"),
                pd.read_csv("../data/raw/sackmann_atp_matches_2023.csv"),
                pd.read_csv("../data/raw/sackmann_atp_matches_2022.csv"),
                pd.read_csv("../data/raw/sackmann_atp_matches_2021.csv"),
                pd.read_csv("../data/raw/sackmann_atp_matches_2020.csv"),
                pd.read_csv("../data/raw/sackmann_atp_matches_2019.csv")]).sort_values(["tourney_date","tourney_name","match_num"])

In [27]:
df["day_num"] = df.groupby(["tourney_name","tourney_date", "winner_name"]).cumcount()+1
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
df['match_date'] = (pd.to_datetime(df['tourney_date']) + pd.to_timedelta(df["day_num"],unit='D')).dt.strftime('%Y-%m-%d')
df.sort_values(["tourney_date","tourney_name","match_num"],inplace=True)
df["competition_name"] = df["tourney_name"]
df["event_id"] = df["tourney_id"].astype(str) + df["match_num"].astype(str)
df["surface"] = df["surface"].str.lower()

In [28]:
df

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,day_num,match_date,competition_name,event_id
26,2019-M020,Brisbane,hard,32,A,2018-12-31,271,106045,NaN,NaN,...,9.0,10.0,63.0,810.0,49.0,974.0,1,2019-01-01,Brisbane,2019-M020271
25,2019-M020,Brisbane,hard,32,A,2018-12-31,272,105357,NaN,NaN,...,13.0,18.0,38.0,1083.0,61.0,814.0,1,2019-01-01,Brisbane,2019-M020272
24,2019-M020,Brisbane,hard,32,A,2018-12-31,273,105777,6.0,NaN,...,0.0,3.0,19.0,1835.0,75.0,701.0,1,2019-01-01,Brisbane,2019-M020273
23,2019-M020,Brisbane,hard,32,A,2018-12-31,275,106034,NaN,Q,...,0.0,1.0,185.0,275.0,102.0,572.0,1,2019-01-01,Brisbane,2019-M020275
22,2019-M020,Brisbane,hard,32,A,2018-12-31,276,104871,NaN,NaN,...,4.0,6.0,40.0,1050.0,57.0,875.0,1,2019-01-01,Brisbane,2019-M020276
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2805,2024-7696,Next Gen Finals,hard,8,F,2024-12-18,396,210530,5.0,NaN,...,4.0,6.0,122.0,493.0,20.0,2355.0,2,2024-12-20,Next Gen Finals,2024-7696396
2804,2024-7696,Next Gen Finals,hard,8,F,2024-12-18,397,209950,1.0,NaN,...,4.0,7.0,20.0,2355.0,48.0,1136.0,1,2024-12-19,Next Gen Finals,2024-7696397
2803,2024-7696,Next Gen Finals,hard,8,F,2024-12-18,398,210530,5.0,NaN,...,5.0,9.0,122.0,493.0,41.0,1245.0,3,2024-12-21,Next Gen Finals,2024-7696398
2802,2024-7696,Next Gen Finals,hard,8,F,2024-12-18,399,211663,8.0,NaN,...,10.0,14.0,145.0,409.0,128.0,471.0,4,2024-12-22,Next Gen Finals,2024-7696399


In [29]:

def parse_score(score):
    sets = score.split(" ")
    winner_sets_won_ = 0
    loser_sets_won_ = 0
    winner_games_won_ = 0
    loser_games_won_ = 0
    winner_tiebreaks_won_ = 0
    loser_tiebreaks_won_ = 0

    try:
        winner_set1_games_ = int(sets[0].split("-")[0])
        loser_set1_games_ = int(re.sub("\(.*\)","",sets[0].split("-")[1]))
        winner_games_won_ += winner_set1_games_
        loser_games_won_ += loser_set1_games_
        if winner_set1_games_ > loser_set1_games_:
            winner_set1_win_flag_ = 1
            loser_set1_win_flag_ = 0
            winner_sets_won_ += 1
        else:
            winner_set1_win_flag_ = 0
            loser_set1_win_flag_ = 1
            loser_sets_won_ += 1
        if winner_set1_games_ == 7 and loser_set1_games_ == 6:
            winner_tiebreaks_won_ += 1
        if winner_set1_games_ == 6 and loser_set1_games_ == 7:
            loser_tiebreaks_won_ += 1
    except:
        winner_set1_games_ = np.nan
        loser_set1_games_ = np.nan
        winner_set1_win_flag_ = np.nan
        loser_set1_win_flag_ = np.nan
    try:
        winner_set2_games_ = int(sets[1].split("-")[0])
        loser_set2_games_ = int(re.sub("\(.*\)","",sets[1].split("-")[1]))
        winner_games_won_ += winner_set2_games_
        loser_games_won_ += loser_set2_games_
        if winner_set2_games_ > loser_set2_games_:
            winner_set2_win_flag_ = 1
            loser_set2_win_flag_ = 0
            winner_sets_won_ += 1
        else:
            winner_set2_win_flag_ = 0
            loser_set2_win_flag_ = 1
            loser_sets_won_ += 1
        if winner_set2_games_ == 7 and loser_set2_games_ == 6:
            winner_tiebreaks_won_ += 1
        if winner_set2_games_ == 6 and loser_set2_games_ == 7:
            loser_tiebreaks_won_ += 1
    except:
        winner_set2_games_ = np.nan
        loser_set2_games_ = np.nan
        winner_set2_win_flag_ = np.nan
        loser_set2_win_flag_ = np.nan
    try:
        winner_set3_games_ = int(sets[2].split("-")[0])
        loser_set3_games_ = int(re.sub("\(.*\)","",sets[2].split("-")[1]))
        winner_games_won_ += winner_set3_games_
        loser_games_won_ += loser_set3_games_
        if winner_set3_games_ > loser_set3_games_:
            winner_set3_win_flag_ = 1
            loser_set3_win_flag_ = 0
            winner_sets_won_ += 1
        else:
            winner_set3_win_flag_ = 0
            loser_set3_win_flag_ = 1
            loser_sets_won_ += 1
        if winner_set3_games_ == 7 and loser_set3_games_ == 6:
            winner_tiebreaks_won_ += 1
        if winner_set3_games_ == 6 and loser_set3_games_ == 7:
            loser_tiebreaks_won_ += 1
    except:
        winner_set3_games_ = np.nan
        loser_set3_games_ = np.nan
        winner_set3_win_flag_ = np.nan
        loser_set3_win_flag_ = np.nan
    try:
        winner_set4_games_ = int(sets[3].split("-")[0])
        loser_set4_games_ = int(re.sub("\(.*\)","",sets[3].split("-")[1]))
        winner_games_won_ += winner_set4_games_
        loser_games_won_ += loser_set4_games_
        if winner_set4_games_ > loser_set4_games_:
            winner_set4_win_flag_ = 1
            loser_set4_win_flag_ = 0
            winner_sets_won_ += 1
        else:
            winner_set4_win_flag_ = 0
            loser_set4_win_flag_ = 1
            loser_sets_won_ += 1
        if winner_set4_games_ == 7 and loser_set4_games_ == 6:
            winner_tiebreaks_won_ += 1
        if winner_set4_games_ == 6 and loser_set4_games_ == 7:
            loser_tiebreaks_won_ += 1
    except:
        winner_set4_games_ = np.nan
        loser_set4_games_ = np.nan
        winner_set4_win_flag_ = np.nan
        loser_set4_win_flag_ = np.nan
    try:
        winner_set5_games_ = int(sets[4].split("-")[0])
        loser_set5_games_ = int(re.sub("\(.*\)","",sets[4].split("-")[1]))
        winner_games_won_ += winner_set5_games_
        loser_games_won_ += loser_set5_games_
        if winner_set5_games_ > loser_set5_games_:
            winner_set5_win_flag_ = 1
            loser_set5_win_flag_ = 0
            winner_sets_won_ += 1
        else:
            winner_set5_win_flag_ = 0
            loser_set5_win_flag_ = 1
            loser_sets_won += 1
        if winner_set5_games_ == 7 and loser_set5_games_ == 6:
            winner_tiebreaks_won_ += 1
        if winner_set5_games_ == 6 and loser_set5_games_ == 7:
            loser_tiebreaks_won_ += 1
    except:
        winner_set5_games_ = np.nan
        loser_set5_games_ = np.nan
        winner_set5_win_flag_ = np.nan
        loser_set5_win_flag_ = np.nan


    return winner_games_won_, loser_games_won_, winner_sets_won_, loser_sets_won_,winner_set1_games_,loser_set1_games_,winner_set2_games_,loser_set2_games_,winner_set3_games_,loser_set3_games_,winner_set4_games_,loser_set4_games_,winner_set5_games_,loser_set5_games_,winner_set1_win_flag_,loser_set1_win_flag_,winner_set2_win_flag_,loser_set2_win_flag_,winner_set3_win_flag_,loser_set3_win_flag_,winner_set4_win_flag_,loser_set4_win_flag_,winner_set5_win_flag_,loser_set5_win_flag_,winner_tiebreaks_won_,loser_tiebreaks_won_



In [30]:
#prepare final dataframe lists
date_ = []
event_id = []
surface = []
comp_names = []
comp_categories = []
player_names = []
player_seeds = []
opponent_names = []
opponent_seeds = []
winner_flag = []
aces = []
breakpoints_won  = []
total_breakpoints  = []
double_faults  = []
first_serve_points_won  = []
first_serve_successful  = []
games_won  = []
service_games = []
service_games_held = []
return_games = []
return_games_broke = []
games_won_= []
games_lost_= []
sets_won_=[]
sets_lost_=[]
tiebreaks_won = []
tiebreaks_lost = []
set1_win_flag_ = []
set2_win_flag_ = []
set3_win_flag_ = []
set4_win_flag_ = []
set5_win_flag_ = []
set1_games_ = []
set2_games_ = []
set3_games_ = []
set4_games_ = []
set5_games_ = []
second_serve_points_won  = []
second_serve_successful  = []

games_spread = []
sets_spread = []
aces_allowed = []
breakpoints_allowed = []
breakpoints_saved = []
first_serve_return_points_won = []
first_serve_return_points_total = []
second_serve_return_points_won = []
second_serve_return_points_total = []

for i in range(len(df.index)):
    row = df.iloc[i,:]
    date_.append(row["match_date"])
    date_.append(row["match_date"])
    surface.append(row["surface"])
    surface.append(row["surface"])
    event_id.append(row['event_id'])
    event_id.append(row['event_id'])
    comp_names.append(row["tourney_name"])
    comp_names.append(row["tourney_name"])
    comp_categories.append("ATP")
    comp_categories.append("ATP")
    player_names.append(row["winner_name"])
    opponent_names.append(row["loser_name"])
    player_names.append(row["loser_name"])
    opponent_names.append(row["winner_name"])
    player_seeds.append(row["winner_seed"])
    opponent_seeds.append(row["loser_seed"])
    player_seeds.append(row["loser_seed"])
    opponent_seeds.append(row["winner_seed"])
    winner_flag.append(1)
    winner_flag.append(0)
    aces.append(row["w_ace"])
    aces.append(row["l_ace"])
    breakpoints_won.append(row["l_bpFaced"] - row["l_bpSaved"])
    breakpoints_won.append(row["w_bpFaced"] - row["w_bpSaved"])
    total_breakpoints.append(row["l_bpFaced"])
    total_breakpoints.append(row["w_bpFaced"])
    double_faults.append(row["w_df"])
    double_faults.append(row["l_df"])
    first_serve_points_won.append(row["w_1stWon"])
    first_serve_points_won.append(row["l_1stWon"])
    first_serve_successful.append(row["w_1stIn"])
    first_serve_successful.append(row["l_1stIn"])
    service_games.append(row["w_SvGms"])
    service_games_held.append(row["w_SvGms"] - (row["w_bpFaced"] - row["w_bpSaved"]))
    assert service_games_held[-1] or 0 <= service_games[-1] or 0
    service_games.append(row["l_SvGms"])
    service_games_held.append(row["l_SvGms"] - (row["l_bpFaced"] - row["l_bpSaved"]))
    assert service_games_held[-1] or 0 <= service_games[-1] or 0
    return_games.append(row["l_SvGms"])
    return_games_broke.append(row["l_SvGms"]-(row["l_bpFaced"]-row["l_bpSaved"]))
    assert return_games_broke[-1] or 0 <= return_games[-1] or 0
    return_games.append(row["w_SvGms"])
    return_games_broke.append(row["w_SvGms"]-(row["w_bpFaced"]-row["w_bpSaved"]))
    assert return_games_broke[-1] or 0 <= return_games[-1] or 0

    score = row["score"]

    winner_games_won_,loser_games_won_,winner_sets_won_,loser_sets_won_,winner_set1_games_,loser_set1_games_,winner_set2_games_,loser_set2_games_,winner_set3_games_,loser_set3_games_,winner_set4_games_,loser_set4_games_,winner_set5_games_,loser_set5_games_,winner_set1_win_flag_,loser_set1_win_flag_,winner_set2_win_flag_,loser_set2_win_flag_,winner_set3_win_flag_,loser_set3_win_flag_,winner_set4_win_flag_,loser_set4_win_flag_,winner_set5_win_flag_,loser_set5_win_flag_,winner_tiebreaks_won_,loser_tiebreaks_won_ = parse_score(score)
    
    sets_won_.append(winner_sets_won_)
    sets_lost_.append(loser_sets_won_)
    sets_won_.append(loser_sets_won_)
    sets_lost_.append(winner_sets_won_)
    sets_spread.append(winner_sets_won_ - loser_sets_won_)
    sets_spread.append(loser_sets_won_ - winner_sets_won_)

    games_won_.append(winner_games_won_)
    games_lost_.append(loser_games_won_)
    games_won_.append(loser_games_won_)
    games_lost_.append(winner_games_won_)
    games_spread.append(winner_games_won_ - loser_games_won_)
    games_spread.append(loser_games_won_ - winner_games_won_)
    
    set1_games_.append(winner_set1_games_)
    set1_games_.append(loser_set1_games_)
    set1_win_flag_.append(winner_set1_win_flag_)
    set1_win_flag_.append(loser_set1_win_flag_)
    
    set2_games_.append(winner_set2_games_)
    set2_games_.append(loser_set2_games_)
    set2_win_flag_.append(winner_set2_win_flag_)
    set2_win_flag_.append(loser_set2_win_flag_)

    set3_games_.append(winner_set3_games_)
    set3_games_.append(loser_set3_games_)
    set3_win_flag_.append(winner_set3_win_flag_)
    set3_win_flag_.append(loser_set3_win_flag_)

    set4_games_.append(winner_set4_games_)
    set4_games_.append(loser_set4_games_)
    set4_win_flag_.append(winner_set4_win_flag_)
    set4_win_flag_.append(loser_set4_win_flag_)

    set5_games_.append(winner_set5_games_)
    set5_games_.append(loser_set5_games_)
    set5_win_flag_.append(winner_set5_win_flag_)
    set5_win_flag_.append(loser_set5_win_flag_)

    tiebreaks_won.append(winner_tiebreaks_won_)
    tiebreaks_won.append(loser_tiebreaks_won_)
    tiebreaks_lost.append(loser_tiebreaks_won_)
    tiebreaks_lost.append(winner_tiebreaks_won_)
    
    second_serve_points_won.append(row["w_2ndWon"])
    second_serve_points_won.append(row["l_2ndWon"])

    second_serve_successful.append(row["w_svpt"] - row["w_df"] - row["w_1stIn"])
    second_serve_successful.append(row["l_svpt"] - row["l_df"] - row["l_1stIn"])

    aces_allowed.append(row["l_ace"])
    aces_allowed.append(row["w_ace"])

    breakpoints_allowed.append(row["w_bpFaced"])
    breakpoints_allowed.append(row["l_bpFaced"])

    breakpoints_saved.append(row["w_bpSaved"])
    breakpoints_saved.append(row["l_bpSaved"])

    first_serve_return_points_won.append(row["l_1stIn"]-row["l_1stWon"])
    first_serve_return_points_won.append(row["w_1stIn"]-row["w_1stWon"])

    first_serve_return_points_total.append(row["l_1stIn"])
    first_serve_return_points_total.append(row["w_1stIn"])
    

    second_serve_return_points_won.append((row["l_svpt"] - row["l_df"] - row["l_1stIn"])-(row["l_2ndWon"]))
    second_serve_return_points_won.append((row["w_svpt"] - row["w_df"] - row["w_1stIn"])-(row["w_2ndWon"]))

    second_serve_return_points_total.append(row["l_svpt"] - row["l_df"] - row["l_1stIn"])
    second_serve_return_points_total.append(row["w_svpt"] - row["w_df"] - row["w_1stIn"])



In [31]:
match_stats_df = pd.DataFrame({"match_date":date_,
"event_id":event_id,
"surface":surface,
"competition_name":comp_names,
"competition_category":comp_categories,
"player_name":player_names,
"opponent_name":opponent_names,
"player_seed":player_seeds,
"opponent_seed":opponent_seeds,
"winner_flag":winner_flag,
"aces":aces,
"aces_allowed":aces_allowed,
"breakpoints_won":breakpoints_won,
"double_faults":double_faults,
"first_serve_points_won":first_serve_points_won,
"first_serve_successful":first_serve_successful,
"service_games":service_games,
"service_games_held":service_games_held,
"return_games":return_games,
"return_games_broke":return_games_broke,
"tiebreaks_won":tiebreaks_won,
"tiebreaks_lost":tiebreaks_lost,
"games_won":games_won_,
"games_lost":games_lost_,
"sets_won":sets_won_,
"sets_lost":sets_lost_,
"second_serve_points_won":second_serve_points_won,
"second_serve_successful":second_serve_successful,
"total_breakpoints":total_breakpoints,
"games_spread":games_spread,
"sets_spread":sets_spread,
"breakpoints_allowed":breakpoints_allowed,
"breakpoints_saved":breakpoints_saved,
"first_serve_return_points_won":first_serve_return_points_won,
"first_serve_return_points_total":first_serve_return_points_total,
"second_serve_return_points_won":second_serve_return_points_won,
"second_serve_return_points_total":second_serve_return_points_total,
"set1_win_flag":set1_win_flag_,
"set2_win_flag":set2_win_flag_,
"set3_win_flag":set3_win_flag_,
"set4_win_flag":set4_win_flag_,
"set5_win_flag":set5_win_flag_,
"set1_games":set1_games_,
"set2_games":set2_games_,
"set3_games":set3_games_,
"set4_games":set4_games_,
"set5_games":set5_games_
})

match_stats_df["first_serve_win_perc"] = 100*match_stats_df["first_serve_points_won"]/match_stats_df["first_serve_successful"]
match_stats_df["first_serve_return_win_perc"] = 100*match_stats_df["first_serve_return_points_won"]/match_stats_df["first_serve_return_points_total"]
match_stats_df["second_serve_win_perc"] = 100*(match_stats_df["second_serve_points_won"]/(match_stats_df["second_serve_successful"] + match_stats_df["double_faults"]))
match_stats_df["breakpoint_conversion_perc"] = 100*(match_stats_df["breakpoints_won"]/(match_stats_df["total_breakpoints"]))
match_stats_df["Month"] = pd.to_datetime(match_stats_df["match_date"]).dt.strftime("%B") + " " + pd.to_datetime(match_stats_df["match_date"]).dt.strftime("%Y")
match_stats_df["Year"] = pd.to_datetime(match_stats_df["match_date"]).dt.strftime("%Y")



In [32]:
name_fix = {'Adolfo Daniel Vallejo':'Vallejo, Adolfo Daniel',
 'Adria Soriano Barrera':'Soriano Barrera, Adria',
 'Aisam Ul Haq Qureshi':'Qureshi, Ahmad Nael',
 'Alan Fernando Rubio Fierros':'Rubio Fierros, Alan Fernando',
 'Alejandro Davidovich Fokina':'Davidovich Fokina, Alejandro',
 'Alejandro Moro Canas':'Moro Canas, Alejandro',
 'Alex De Minaur':'De Minaur, Alex',
 'Alvaro Guillen Meza': 'Guillen Meza, Alvaro',
 'Bernabe Zapata Miralles':'Zapata Miralles, Bernabe',
 'Botic Van De Zandschulp': 'Van De Zandschulp, Botic',
 'Camilo Ugo Carabelli':'Ugo Carabelli, Camilo',
 'Carlos Gimeno Valero':'Gimeno Valero, Carlos',
 'Cedrik Marcel Stebe':'Stebe, Cedrik-Marcel',
 'Chak Lam Coleman Wong':'Coleman, Wong',
 'Chun Hsin Tseng':'Tseng, Chun Hsin',
 'Courtney John Lock':'Lock, Courtney John',
 'Daniel Dutra Da Silva':'Dutra Da Silva, Daniel',
 'Daniel Elahi Galan':'Galan, Daniel Elahi',
 'David Jorda Sanchis':'Jorda Sanchis, David',
 'Diego Fernandez Flores':'Fernandez Flores, Diego',
 'Dragos Nicolae Madaras':"Cazacu, Dragos Nicolae",
 'Duck Hee Lee':"Lee, Duckhee",
 'Facundo Diaz Acosta':'Diaz Acosta, Facundo',
 'Federico Agustin Gomez':'Gomez, Federico Agustin',
 'Felipe Meligeni Alves':'Meligeni Alves, Felipe',
 'Felix Auger Aliassime':'Auger-Aliassime, Felix',
 'Filip Cristian Jianu':'Jianu, Filip Cristian',
 'Frederico Ferreira Silva': 'Ferreira Silva, Frederico',
 'Gabi Adrian Boitan':'Boitan, Gabi Adrian',
 'Genaro Alberto Olivieri':'Olivieri, Genaro Alberto',
 'Gian Marco Moroni':'Moroni, Filippo',
 'Gilles Arnaud Bailly':'Arnaud Bailly, Gilles',
 'Giovanni Mpetshi Perricard':'Mpetshi Perricard, Giovanni',
 'Hernando Jose Escurra Isnardi':'Escurra Isnardi, Hernando Jose',
 'J J Wolf':'Wolf, Jeffrey John',
 'Jan Lennard Struff':'Struff, Jan-Lennard',
 'Javier Barranco Cosano':'Barranco Cosano, Javier',
 'Jesper De Jong':'De Jong, Jesper',
 'Ji Sung Nam':'Nam, Ji Sung',
 'Joaquin  Aguilar Cardozo ':'Aguilar Cardozo, Joaquin',
 'Joris De Loore':'De Loore, Joris',
 'Jose  Flores':'Flores, Jose',
 'Juan Carlos Prado Angelo':'Prado Angelo, Juan Carlos',
 'Juan Manuel Cerundolo':'Cerundolo, Juan Manuel',
 'Juan Martin del Potro':'del Potro, Juan Martin',
 'Juan Pablo Ficovich':'Ficovich, Juan Pablo',
 'Juan Pablo Varillas':'Varillas, Juan Pablo',
 'Karim Mohamed Maamoun':'Maamoun, Karim Mohamed',
 'Kris Van Wyk':'van Wyk, Kris',
 'Liova Ayite Ajavon':'Ajavon, Liova',
 'Luca Van Assche':'Van Assche, Luca',
 'Luis Carlos Alvarez Valdes': 'Alvarez Valdes,Luis Carlos',
 'Luis David Martinez':'Martinez, Luis David',
 'Lukas Hellum Lilleengen':'Hellum Lilleengen, Lukas',
 'Marc Andrea Huesler':'Huesler, Marc-Andrea',
 'Martin Antonio Vergara Del Puerto':'Vergara Del Puerto, Martin Antonio',
 'Matheus Pucinelli De Almeida':'Pucinelli de Almeida, Matheus',
 'Max Hans Rehberg':'Rehberg, Max Hans',
 'Michael Bassem Sobhy':'Bassem Sobhy, Michael',
 'Mubarak Shannan Zayid':'Zayid, Mubarak Shannan',
 'Mustapha El Natour':'El Natour, Mustapha',
 'N Sriram Balaji': 'Balaji, N Sriram',
 'Nathan Anthony Barki':'Barki, Nathan Anthony',
 'Nicholas David Ionel':'Ionel, Nicholas David',
 'Nicolai Budkov Kjaer':'Budkov Kjaer, Nicolai',
 'Nicolas Alvarez Varona':'Alvarez Varona, Nicolas',
 'Nicolas Moreno De Alboran':'Moreno de Alboran, Nicolas',
 'Niki Kaliyanda Poonacha':'Kaliyanda Poonacha, Niki',
 'Nikolas Sanchez Izquierdo':'Sanchez Izquierdo, Nikolas',
 'Oriol Roca Batalla':'Roca Batalla, Oriol',
 'Pablo Carreno Busta':'Carreno Busta, Pablo',
 'Pablo Llamas Ruiz':'Llamas Ruiz, Pablo',
 'Patrik Niklas Salminen':'Niklas-Salminen, Patrik',
 'Phuong Van Nguyen':'Phuong Van Nguyen',
 'Pierre Hugues Herbert':'Herbert, Pierre-Hugues',
 'Roberto Bautista Agut':'Bautista Agut, Roberto',
 'Roberto Carballes Baena':'Carballes Baena, Roberto',
 'Rodrigo Pacheco Mendez':'Pacheco Mendez, Rodrigo',
 'Roman Andres Burruchaga':'Burruchaga, Roman Andres',
 'Santiago Fa Rodriguez Taverna':'Rodriguez Taverna, Santiago',
 'Sasi Kumar Mukund':'Mukund, Sasikumar',
 'Soon Woo Kwon':'Kwon, Soonwoo',
 'Tegar Abdi Satrio Wibowo':'Wibowo, Tegar Abdi Satrio',
 'Thai Son Kwiatkowski':'Kwiatkowski, Thai-Son',
 'Thiago Agustin Tirante':'Tirante, Thiago Agustin',
 'Thiago Seyboth Wild':'Seyboth Wild, Thiago',
 'Thiemo De Bakker':'De Bakker, Thiemo',
 'Thomas Yaka Kofi Setodji':'Setodji, Thomas',
 'Tim Van Rijthoven':'Van Rijthoven, Tim',
 'Tomas Barrios Vera':'Barrios Vera, Marcelo Tomas',
 'Tomas Martin Etcheverry':'Etcheverry, Tomas Martin',
 'Tung Lin Wu':'Wu, Tung-Lin',
 'Yasitha De Silva':'De Silva, Yasitha',
 'Ye Cong Mo':'Mo, Ye Cong',
 'Younes Lalami Laaroussi':'Lalami Laaroussi, Younes',
 'Yu Hsiou Hsu':'Hsu, Yu Hsiou'}
player_name2 = []
anomalies = []
for i in match_stats_df.index:
    pn = match_stats_df.loc[i,"player_name"]
    pn_split = pn.split(" ")
    if len(pn_split) == 2:
        player_name2.append(pn_split[1]+", "+pn_split[0])
    elif pn in name_fix:
        player_name2.append(name_fix[pn])   
        
    else:
        anomalies.append(pn)
        player_name2.append(pn)
match_stats_df["player_name2"] = player_name2


opponent_name2 = []
anomalies = []
for i in match_stats_df.index:
    pn = match_stats_df.loc[i,"opponent_name"]
    pn_split = pn.split(" ")
    if len(pn_split) == 2:
        opponent_name2.append(pn_split[1]+", "+pn_split[0])
    elif pn in name_fix:
        opponent_name2.append(name_fix[pn])   
        
    else:
        anomalies.append(pn)
        opponent_name2.append(pn)
match_stats_df["opponent_name2"] = opponent_name2

In [33]:
player_names = pd.read_csv("../data/prep/match_stats.csv")
player_names = player_names[~player_names.player_name.str.contains("/")].player_name.unique()

def find_fuzzy_match(query, choices, cutoff=80):
    if len(query.split(" ")) == 2:
        return query
    # Use extractOne to find the single best match above the cutoff
    match = process.extractOne(query, choices, scorer=fuzz.token_sort_ratio)
    # Returns a tuple: (matched_string, similarity_score, matched_index)
    if match and match[1] >= cutoff:
        return match[0] # Return the matched string
    else:
        return query # Return NaN if no match is found above threshold

# Apply the fuzzy matching function to the left DataFrame's column
match_stats_df['player_name'] = match_stats_df['player_name2'].apply(lambda x: find_fuzzy_match(x, player_names))
match_stats_df.drop('player_name2',axis=1,inplace=True)

match_stats_df['opponent_name'] = match_stats_df['opponent_name2'].apply(lambda x: find_fuzzy_match(x, player_names))
match_stats_df.drop('opponent_name2',axis=1,inplace=True)

/tmp/ipykernel_10601/1544618979.py:1: DtypeWarning: Columns (4,7,8,10,11,69) have mixed types. Specify dtype option on import or set low_memory=False.
  player_names = pd.read_csv("../data/prep/match_stats.csv")


In [34]:
match_stats_df.to_csv("../data/prep/retro/atp_sackmann.csv")